In [2]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import time
import numpy as np
from loguru import logger
import warnings
warnings.filterwarnings("ignore")

#data loading
csv_basic = '../../Data/important/basic_eth_100d_scam.csv'
csv_scam = '../../Data/important/info_eth_100dscam.csv'
csv_scamliq = '../../Data/important/liq_eth_100d_scam.csv'
csv_rugpullseller = '../../Data/important/analysis/rugpull_sellers.csv'
csv_unique_wallet = '../../Data/important/analysis/unique_wallets.csv'


df_scaminfo = pd.read_csv(csv_scam)
df_scamliq = pd.read_csv(csv_scamliq)
df_rugpullseller = pd.read_csv(csv_rugpullseller)
df_unique_wallet = pd.read_csv(csv_unique_wallet)
df_single_seller = pd.read_csv("../../Data/important/analysis/single_sellerinfo.csv")
df_multi_seller = pd.read_csv("../../Data/important/analysis/multiseller_info.csv")


pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

print('Done data loading')

Done data loading


# Data capturing of time

In [4]:
import pandas as pd

# ---------- 1) Prep scamliq and aggregate per (sender, pool) ----------
liq = df_scamliq.copy()
liq['timestamp']     = pd.to_datetime(liq['timestamp'], utc=True, errors='coerce')
liq['seller_norm']   = liq['sender_address'].astype(str).str.lower().str.strip()
liq['pool_norm']     = liq['pool_address'].astype(str).str.lower().str.strip()
liq['category_norm'] = liq['category'].astype(str).str.lower().str.strip()

sells = liq[liq['category_norm'].eq('sell')].dropna(subset=['timestamp'])

per_sender_pool = (
    sells.groupby(['seller_norm','pool_norm'])['timestamp']
         .agg(first_sell_time='min',
              last_sell_time='max',
              sell_count='size')
         .reset_index()
)

# ---------- 2) Prep df_single_seller keys and merge on (seller, pool) ----------
single = df_single_seller.copy()

# normalize join keys
single['seller_norm'] = single['seller'].astype(str).str.lower().str.strip()
single['pool_norm']   = single['pool_address'].astype(str).str.lower().str.strip()

# ensure created time is parsed
single['pool_created_time'] = pd.to_datetime(single['pool_created_time'], utc=True, errors='coerce')

# merge stats back
single = single.merge(per_sender_pool,
                      how='left',
                      on=['seller_norm','pool_norm'])

# ---------- 3) (Optional) sanity: require first sell to be after creation ----------
valid = (
    single['first_sell_time'].notna() &
    single['pool_created_time'].notna() &
    (single['first_sell_time'] >= single['pool_created_time'])
)

single.loc[~valid, ['first_sell_time','last_sell_time']] = pd.NaT
single['sell_count'] = single['sell_count'].fillna(0).astype('Int64')

# optional convenience metrics
single['first_sell_delay'] = single['first_sell_time'] - single['pool_created_time']
single['last_sell_delay']  = single['last_sell_time']  - single['pool_created_time']
single['sell_span']        = single['last_sell_time']  - single['first_sell_time']

# cleanup helper columns (if you don’t need them later)
single = single.drop(columns=['seller_norm','pool_norm'])


In [5]:
single

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,pool_address,rugpull_sellers,seller_count,pool_created_time,created_year,token_owner_x,seller_list,seller,tag,token_owner_y,first_sell_time,last_sell_time,sell_count,first_sell_delay,last_sell_delay,sell_span
0,0,0,0,0x4E0E28d426caf318747B8E05C8B0564A580E39a7,"[""0xA8C7372dC993d7510C9c45425807d463967cbb12""]",1,2018-11-02 22:56:11+00:00,2018,0xAd850d65eB5202f828f5f7883bc0B46ac87e64D4,['0xA8C7372dC993d7510C9c45425807d463967cbb12'],0xA8C7372dC993d7510C9c45425807d463967cbb12,single-nonowner,0xAd850d65eB5202f828f5f7883bc0B46ac87e64D4,2018-11-03 10:26:39+00:00,2018-11-03 10:26:39+00:00,4,0 days 11:30:28,0 days 11:30:28,0 days 00:00:00
1,1,1,1,0xbaf5A8BDF81cfE2d34c0CeD89236FE473183F2E8,"[""0x8948E4B00DEB0a5ADb909F4DC5789d20D0851D71""]",1,2019-03-07 05:59:20+00:00,2019,0x8948E4B00DEB0a5ADb909F4DC5789d20D0851D71,['0x8948E4B00DEB0a5ADb909F4DC5789d20D0851D71'],0x8948E4B00DEB0a5ADb909F4DC5789d20D0851D71,single-owner,0x8948E4B00DEB0a5ADb909F4DC5789d20D0851D71,2019-03-07 18:54:32+00:00,2019-03-07 19:09:30+00:00,4,0 days 12:55:12,0 days 13:10:10,0 days 00:14:58
2,2,2,2,0x225026D626E45FA662e6a71F679efF0CAc3054f1,"[""0x006004fFA18E3cf78fA3b50393ec44C1ab89cF6c""]",1,2019-03-11 14:01:50+00:00,2019,0xf1fa9a38914E853DE933FbF7Df2f278701e873DF,['0x006004fFA18E3cf78fA3b50393ec44C1ab89cF6c'],0x006004fFA18E3cf78fA3b50393ec44C1ab89cF6c,single-nonowner,0xf1fa9a38914E853DE933FbF7Df2f278701e873DF,2019-03-12 07:24:20+00:00,2019-03-12 07:24:20+00:00,2,0 days 17:22:30,0 days 17:22:30,0 days 00:00:00
3,3,3,3,0x9394C20adca4512DfC3d3c184c648E4193462Ebb,"[""0x2523C15dB0e3843DfC7C08772c8331Fb40CC8a0F""]",1,2019-04-22 08:02:06+00:00,2019,0x2523C15dB0e3843DfC7C08772c8331Fb40CC8a0F,['0x2523C15dB0e3843DfC7C08772c8331Fb40CC8a0F'],0x2523C15dB0e3843DfC7C08772c8331Fb40CC8a0F,single-owner,0x2523C15dB0e3843DfC7C08772c8331Fb40CC8a0F,2019-04-22 18:16:37+00:00,2019-04-25 05:33:33+00:00,8,0 days 10:14:31,2 days 21:31:27,2 days 11:16:56
4,4,4,4,0xEda88dDb13888C9A4dE7304965E9315E69ea980E,"[""0x866cb16F2162c319E48351827A7e15DDf0405E16""]",1,2019-06-10 00:06:22+00:00,2019,0x866cb16F2162c319E48351827A7e15DDf0405E16,['0x866cb16F2162c319E48351827A7e15DDf0405E16'],0x866cb16F2162c319E48351827A7e15DDf0405E16,single-owner,0x866cb16F2162c319E48351827A7e15DDf0405E16,2019-06-14 02:09:24+00:00,2019-06-14 20:53:38+00:00,4,4 days 02:03:02,4 days 20:47:16,0 days 18:44:14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29568,29568,29568,29568,0x031D046Cbd8727D15702ECd5299eCcd63ADE84E8,"[""0xC14664811a2a4c233d253fDD03dee4B97ABBEbb5""]",1,2024-11-30 13:18:47+00:00,2024,0xCC54E2644ABb3B02EDb21aCD8F4f9ee06432Cada,['0xC14664811a2a4c233d253fDD03dee4B97ABBEbb5'],0xC14664811a2a4c233d253fDD03dee4B97ABBEbb5,single-nonowner,0xCC54E2644ABb3B02EDb21aCD8F4f9ee06432Cada,2024-12-01 17:50:23+00:00,2024-12-01 17:50:23+00:00,2,1 days 04:31:36,1 days 04:31:36,0 days 00:00:00
29569,29569,29569,29569,0xd98d929EBc79856BBaEB783561294999c424211B,"[""0x670C625612D1662c91dc7B4434207101e7bd941f""]",1,2024-11-30 20:59:23+00:00,2024,0x670C625612D1662c91dc7B4434207101e7bd941f,['0x670C625612D1662c91dc7B4434207101e7bd941f'],0x670C625612D1662c91dc7B4434207101e7bd941f,single-owner,0x670C625612D1662c91dc7B4434207101e7bd941f,2024-12-01 08:54:11+00:00,2024-12-01 08:54:11+00:00,2,0 days 11:54:48,0 days 11:54:48,0 days 00:00:00
29570,29570,29570,29570,0x6E7FE3428816cbD1CB1788aa84d04C4a08248b02,"[""0x670C625612D1662c91dc7B4434207101e7bd941f""]",1,2024-11-30 21:41:47+00:00,2024,0x670C625612D1662c91dc7B4434207101e7bd941f,['0x670C625612D1662c91dc7B4434207101e7bd941f'],0x670C625612D1662c91dc7B4434207101e7bd941f,single-owner,0x670C625612D1662c91dc7B4434207101e7bd941f,2024-12-01 08:56:11+00:00,2024-12-01 08:56:11+00:00,2,0 days 11:14:24,0 days 11:14:24,0 days 00:00:00
29571,29571,29571,29571,0x30600e7a89405e5bD76Be94dABB54d776F93E726,"[""0x3328F7f4A1D1C57c35df56bBf0c9dCAFCA309C49""]",1,2024-11-30 22:27:59+00:00,2024,0xEC055397730484b73a9308d24A2A365c86182804,['0x3328F7f4A1

In [12]:
# copy sell stats (first_sell_time, last_sell_time, sell_count) from `single`
cols = ['pool_address', 'first_sell_time', 'last_sell_time', 'sell_count', 'first_sell_delay', 'last_sell_delay', 'sell_span']
sell_stats = single[cols].copy()

# ensure datetimes are proper
sell_stats['first_sell_time'] = pd.to_datetime(sell_stats['first_sell_time'], utc=True, errors='coerce')
sell_stats['last_sell_time'] = pd.to_datetime(sell_stats['last_sell_time'], utc=True, errors='coerce')
# sell_stats['first_sell_delay'] = pd.to_datetime(sell_stats['first_sell_delay'], utc=True, errors='coerce')
# sell_stats['last_sell_delay'] = pd.to_datetime(sell_stats['last_sell_delay'], utc=True, errors='coerce')



# merge into the original df_single_seller by pool_address (left join to preserve original rows)
df_single_seller = df_single_seller.merge(sell_stats, on='pool_address', how='left')

# show result
df_single_seller.head()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,pool_address,rugpull_sellers,seller_count,pool_created_time,created_year,token_owner_x,seller_list,seller,tag,token_owner_y,first_sell_time,last_sell_time,sell_count,first_sell_delay,last_sell_delay,sell_span
0,0,0,0,0x4E0E28d426caf318747B8E05C8B0564A580E39a7,"[""0xA8C7372dC993d7510C9c45425807d463967cbb12""]",1,2018-11-02 22:56:11,2018,0xAd850d65eB5202f828f5f7883bc0B46ac87e64D4,['0xA8C7372dC993d7510C9c45425807d463967cbb12'],0xA8C7372dC993d7510C9c45425807d463967cbb12,single-nonowner,0xAd850d65eB5202f828f5f7883bc0B46ac87e64D4,2018-11-03 10:26:39+00:00,2018-11-03 10:26:39+00:00,4,0 days 11:30:28,0 days 11:30:28,0 days 00:00:00
1,1,1,1,0xbaf5A8BDF81cfE2d34c0CeD89236FE473183F2E8,"[""0x8948E4B00DEB0a5ADb909F4DC5789d20D0851D71""]",1,2019-03-07 05:59:20,2019,0x8948E4B00DEB0a5ADb909F4DC5789d20D0851D71,['0x8948E4B00DEB0a5ADb909F4DC5789d20D0851D71'],0x8948E4B00DEB0a5ADb909F4DC5789d20D0851D71,single-owner,0x8948E4B00DEB0a5ADb909F4DC5789d20D0851D71,2019-03-07 18:54:32+00:00,2019-03-07 19:09:30+00:00,4,0 days 12:55:12,0 days 13:10:10,0 days 00:14:58
2,2,2,2,0x225026D626E45FA662e6a71F679efF0CAc3054f1,"[""0x006004fFA18E3cf78fA3b50393ec44C1ab89cF6c""]",1,2019-03-11 14:01:50,2019,0xf1fa9a38914E853DE933FbF7Df2f278701e873DF,['0x006004fFA18E3cf78fA3b50393ec44C1ab89cF6c'],0x006004fFA18E3cf78fA3b50393ec44C1ab89cF6c,single-nonowner,0xf1fa9a38914E853DE933FbF7Df2f278701e873DF,2019-03-12 07:24:20+00:00,2019-03-12 07:24:20+00:00,2,0 days 17:22:30,0 days 17:22:30,0 days 00:00:00
3,3,3,3,0x9394C20adca4512DfC3d3c184c648E4193462Ebb,"[""0x2523C15dB0e3843DfC7C08772c8331Fb40CC8a0F""]",1,2019-04-22 08:02:06,2019,0x2523C15dB0e3843DfC7C08772c8331Fb40CC8a0F,['0x2523C15dB0e3843DfC7C08772c8331Fb40CC8a0F'],0x2523C15dB0e3843DfC7C08772c8331Fb40CC8a0F,single-owner,0x2523C15dB0e3843DfC7C08772c8331Fb40CC8a0F,2019-04-22 18:16:37+00:00,2019-04-25 05:33:33+00:00,8,0 days 10:14:31,2 days 21:31:27,2 days 11:16:56
4,4,4,4,0xEda88dDb13888C9A4dE7304965E9315E69ea980E,"[""0x866cb16F2162c319E48351827A7e15DDf0405E16""]",1,2019-06-10 00:06:22,2019,0x866cb16F2162c319E48351827A7e15DDf0405E16,['0x866cb16F2162c319E48351827A7e15DDf0405E16'],0x866cb16F2162c319E48351827A7e15DDf0405E16,single-owner,0x866cb16F2162c319E48351827A7e15DDf0405E16,2019-06-14 02:09:24+00:00,2019-06-14 20:53:38+00:00,4,4 days 02:03:02,4 days 20:47:16,0 days 18:44:14


In [ ]:
df_single_seller

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,pool_address,rugpull_sellers,seller_count,pool_created_time,created_year,token_owner_x,seller_list,seller,tag,token_owner_y
0,0,0,0,0x4E0E28d426caf318747B8E05C8B0564A580E39a7,"[""0xA8C7372dC993d7510C9c45425807d463967cbb12""]",1,2018-11-02 22:56:11,2018,0xAd850d65eB5202f828f5f7883bc0B46ac87e64D4,['0xA8C7372dC993d7510C9c45425807d463967cbb12'],0xA8C7372dC993d7510C9c45425807d463967cbb12,single-nonowner,0xAd850d65eB5202f828f5f7883bc0B46ac87e64D4
1,1,1,1,0xbaf5A8BDF81cfE2d34c0CeD89236FE473183F2E8,"[""0x8948E4B00DEB0a5ADb909F4DC5789d20D0851D71""]",1,2019-03-07 05:59:20,2019,0x8948E4B00DEB0a5ADb909F4DC5789d20D0851D71,['0x8948E4B00DEB0a5ADb909F4DC5789d20D0851D71'],0x8948E4B00DEB0a5ADb909F4DC5789d20D0851D71,single-owner,0x8948E4B00DEB0a5ADb909F4DC5789d20D0851D71
2,2,2,2,0x225026D626E45FA662e6a71F679efF0CAc3054f1,"[""0x006004fFA18E3cf78fA3b50393ec44C1ab89cF6c""]",1,2019-03-11 14:01:50,2019,0xf1fa9a38914E853DE933FbF7Df2f278701e873DF,['0x006004fFA18E3cf78fA3b50393ec44C1ab89cF6c'],0x006004fFA18E3cf78fA3b50393ec44C1ab89cF6c,single-nonowner,0xf1fa9a38914E853DE933FbF7Df2f278701e873DF
3,3,3,3,0x9394C20adca4512DfC3d3c184c648E4193462Ebb,"[""0x2523C15dB0e3843DfC7C08772c8331Fb40CC8a0F""]",1,2019-04-22 08:02:06,2019,0x2523C15dB0e3843DfC7C08772c8331Fb40CC8a0F,['0x2523C15dB0e3843DfC7C08772c8331Fb40CC8a0F'],0x2523C15dB0e3843DfC7C08772c8331Fb40CC8a0F,single-owner,0x2523C15dB0e3843DfC7C08772c8331Fb40CC8a0F
4,4,4,4,0xEda88dDb13888C9A4dE7304965E9315E69ea980E,"[""0x866cb16F2162c319E48351827A7e15DDf0405E16""]",1,2019-06-10 00:06:22,2019,0x866cb16F2162c319E48351827A7e15DDf0405E16,['0x866cb16F2162c319E48351827A7e15DDf0405E16'],0x866cb16F2162c319E48351827A7e15DDf0405E16,single-owner,0x866cb16F2162c319E48351827A7e15DDf0405E16


In [13]:
df_single_seller.to_csv("../../Data/important/analysis/single_sellerinfo.csv")

In [6]:
import pandas as pd
from ast import literal_eval

# --- 1) Per-(sender, pool) SELL aggregates from df_scamliq (uses 'timestamp') ---
liq = df_scamliq.copy()
liq['timestamp']     = pd.to_datetime(liq['timestamp'], utc=True, errors='coerce')
liq['seller_norm']   = liq['sender_address'].astype(str).str.lower().str.strip()
liq['pool_norm']     = liq['pool_address'].astype(str).str.lower().str.strip()
liq['category_norm'] = liq['category'].astype(str).str.lower().str.strip()

sells = liq[liq['category_norm'].eq('sell')].dropna(subset=['timestamp'])

per_sender_pool = (
    sells.groupby(['seller_norm', 'pool_norm'])['timestamp']
         .agg(first_sell_time='min', last_sell_time='max', sell_count='size')
         .reset_index()
)

# --- 2) Prepare df_mult: normalize pool key, explode wallet list, normalize sellers ---
dfm = df_multi_seller.copy()
dfm['row_id'] = dfm.index

# parse list if it's a stringified list
def to_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            v = literal_eval(x)
            return v if isinstance(v, list) else [x]
        except Exception:
            return [x]
    return [] if pd.isna(x) else [str(x)]

dfm['rugpull_sellers'] = dfm['rugpull_sellers'].apply(to_list)

dfm['pool_norm'] = dfm['pool_address'].astype(str).str.lower().str.strip()
dfm['pool_created_time'] = pd.to_datetime(dfm.get('pool_created_time'), utc=True, errors='coerce')

exploded = (
    dfm[['row_id', 'pool_norm', 'pool_created_time', 'rugpull_sellers']]
      .explode('rugpull_sellers', ignore_index=False)
      .rename(columns={'rugpull_sellers': 'seller'})
)
exploded['seller_norm'] = exploded['seller'].astype(str).str.lower().str.strip()

# avoid double counting if same seller appears twice in a row
exploded = exploded.drop_duplicates(['row_id', 'seller_norm'])

# --- 3) Join per-(seller, pool) stats, then aggregate per original row ---
joined = exploded.merge(per_sender_pool, how='left', on=['seller_norm', 'pool_norm'])

per_row = (
    joined.groupby('row_id', as_index=True)
          .agg(first_sell_time=('first_sell_time', 'min'),
               last_sell_time =('last_sell_time',  'max'),
               sell_count     =('sell_count',      'sum'))
          .fillna({'sell_count': 0})
)

# --- 4) Merge back and compute optional delays/spans ---
out = dfm.merge(per_row, how='left', on='row_id')
out['sell_count'] = out['sell_count'].fillna(0).astype('Int64')

# (Optional) keep only sells occurring AFTER pool creation (row-wise)
mask = (
    out['first_sell_time'].notna() &
    out['pool_created_time'].notna() &
    (out['first_sell_time'] >= out['pool_created_time'])
)
out.loc[~mask, ['first_sell_time', 'last_sell_time']] = pd.NaT

# Optional timing features
out['first_sell_delay'] = out['first_sell_time'] - out['pool_created_time']
out['last_sell_delay']  = out['last_sell_time']  - out['pool_created_time']
out['sell_span']        = out['last_sell_time']  - out['first_sell_time']

# tidy
out = out.drop(columns=['row_id'])


In [14]:
out

,Unnamed: 0.1,Unnamed: 0,pool_address,rugpull_sellers,seller_count,pool_created_time,created_year,token_owner_x,seller_list,token_owner_y,tag,pool_norm,first_sell_time,last_sell_time,sell_count,first_sell_delay,last_sell_delay,sell_span
0,0,0,0x467fB51D54d7e51eE925F7f1a81AD5f2a0211169,"[0x17559a8138E72C6Ef34512b11685dedC509BaDcf, 0...",2,2018-11-10 03:32:20+00:00,2018,0x00033390560d00f372ae50cD070b8124be5ecE5d,"['0x17559a8138E72C6Ef34512b11685dedC509BaDcf',...",0x00033390560d00f372ae50cD070b8124be5ecE5d,multi_nonowner,0x467fb51d54d7e51ee925f7f1a81ad5f2a0211169,2019-10-21 01:04:02+00:00,2019-10-21 03:23:15+00:00,8,344 days 21:31:42,344 days 23:50:55,0 days 02:19:13
1,1,1,0xC3c028721F854BC75967Cbe432fB0e221908Baa1,"[0x006004fFA18E3cf78fA3b50393ec44C1ab89cF6c, 0...",2,2018-12-09 19:41:39+00:00,2018,0x0ad59C344359Fdf8472E7FFbf4eB6AF4751138DA,"['0x006004fFA18E3cf78fA3b50393ec44C1ab89cF6c',...",0x0ad59C344359Fdf8472E7FFbf4eB6AF4751138DA,multi_nonowner,0xc3c028721f854bc75967cbe432fb0e221908baa1,2019-02-09 23:17:52+00:00,2019-02-14 20:59:54+00:00,12,62 days 03:36:13,67 days 01:18:15,4 days 21:42:02
2,2,2,0x68326300DF49ec6387E75690857424c2ae111750,"[0x52DA601C635951a464DA1E38503b7e9C04d7D530, 0...",2,2018-12-25 08:34:25+00:00,2018,0x5c4c0B176825311452764A2e7327C6b0273b2db2,"['0x52DA601C635951a464DA1E38503b7e9C04d7D530',...",0x5c4c0B176825311452764A2e7327C6b0273b2db2,multi_nonowner,0x68326300df49ec6387e75690857424c2ae111750,2020-05-05 09:13:47+00:00,2020-05-05 19:14:46+00:00,8,497 days 00:39:22,497 days 10:40:21,0 days 10:00:59
3,3,3,0x5d40522c20326F2Ebcec2D371f250e352E3BED27,"[0x35EfCe5f4bd52356e8215BCB9dC7687aDe6B6400, 0...",7,2019-02-11 16:24:49+00:00,2019,0x0aC694A1b86645f2c8563E5fB66a6a22152a694f,"['0x35EfCe5f4bd52356e8215BCB9dC7687aDe6B6400',...",0x0aC694A1b86645f2c8563E5fB66a6a22152a694f,multi_nonowner,0x5d40522c20326f2ebcec2d371f250e352e3bed27,2020-01-23 05:10:44+00:00,2020-02-13 23:58:22+00:00,16,345 days 12:45:55,367 days 07:33:33,21 days 18:47:38
4,4,4,0xCa265A7F4C9dc47b259850B696eBeFFA8BB18d9D,"[0x48143eC3AcA8A255496e8ee7997591D3335Abf5f, 0...",2,2019-05-18 19:59:37+00:00,2019,0x00806578B4B54D41224cfb1568eFB331DD74e25f,"['0x48143eC3AcA8A255496e8ee7997591D3335Abf5f',...",0x00806578B4B54D41224cfb1568eFB331DD74e25f,multi_nonowner,0xca265a7f4c9dc47b259850b696ebeffa8bb18d9d,2020-02-09 18:18:11+00:00,2020-02-11 06:25:22+00:00,4,266 days 22:18:34,268 days 10:25:45,1 days 12:07:11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75856,75856,75856,0x80777279644b44cBA3F5EC23B4AACDf76b106793,"[0x111527f1386c6725a2F5986230f3060BDCAc041F, 0...",45,2024-11-30 19:08:59+00:00,2024,0x55bD4fe4acd12F37bb05Db2bc87Bcfd8D947e15B,"['0x111527f1386c6725a2F5986230f3060BDCAc041F',...",0x55bD4fe4acd12F37bb05Db2bc87Bcfd8D947e15B,multi_nonowner,0x80777279644b44cba3f5ec23b4aacdf76b106793,2024-12-01 06:10:47+00:00,2024-12-07 15:28:35+00:00,272,0 days 11:01:48,6 days 20:19:36,6 days 09:17:48
75857,75857,75857,0x6fFd814BBDd0c7D0Ab4050030f2536EC7AE6B0Ba,"[0x00000000009E50a7dDb7a7B0e2ee6604fd120E49, 0...",30,2024-11-30 20:20:59+00:00,2024,0x9560633fbAC92A84328205E22a6D0cFd0b790296,"['0x00000000009E50a7dDb7a7B0e2ee6604fd120E49',...",0x9560633fbAC92A84328205E22a6D0cFd0b790296,multi_nonowner,0x6ffd814bbdd0c7d0ab4050030f2536ec7ae6b0ba,2024-12-01 07:23:59+00:00,2025-01-29 07:29:47+00:00,286,0 days 11:03:00,59 days 11:08:48,59 days 00:05:48
75858,75858,75858,0xa89f5Fefa693418d6A40ef68366493BaAA5c8212,"[0x000000d40B595B94918a28b27d1e2C66F43A51d3, 0...",50,2024-11-30 23:01:47+00:00,2024,0x22748a8FfCC62940cbAb2f800E39832E16088C6E,"['0x000000d40B595B94918a28b27d1e2C66F43A51d3',...",0x22748a8FfCC62940cbAb2f800E39832E16088C6E,multi_nonowner,0xa89f5fefa693418d6a40ef68366493baaa5c8212,2024-12-01 10:03:11+00:00,2025-03-10 00:51:23+00:00,764,0 days 11:01:24,99 days 01:49:36,98 days 14:48:12
75859,75859,75859,0x9474F4C5b7a491cf223c960Cf52D60B147acBE17,"[0x018f8E759a6F9Eb859F88f89De69F71B93379409, 0...",21,2024-11-30 23:05:59+00:00,2024,0x0E971EFD97F7

In [16]:
out

# copy sell stats (first_sell_time, last_sell_time, sell_count) from `single`
cols = ['pool_address', 'first_sell_time', 'last_sell_time', 'sell_count', 'first_sell_delay', 'last_sell_delay', 'sell_span']
sell_stats = out[cols].copy()

# ensure datetimes are proper
sell_stats['first_sell_time'] = pd.to_datetime(sell_stats['first_sell_time'], utc=True, errors='coerce')
sell_stats['last_sell_time'] = pd.to_datetime(sell_stats['last_sell_time'], utc=True, errors='coerce')

# merge into the original df_single_seller by pool_address (left join to preserve original rows)
df_multi_seller = df_multi_seller.merge(sell_stats, on='pool_address', how='left')

# show result
df_multi_seller.head()

,Unnamed: 0.1,Unnamed: 0,pool_address,rugpull_sellers,seller_count,pool_created_time,created_year,token_owner_x,seller_list,token_owner_y,tag,first_sell_time,last_sell_time,sell_count,first_sell_delay,last_sell_delay,sell_span
0,0,0,0x467fB51D54d7e51eE925F7f1a81AD5f2a0211169,"[""0x17559a8138E72C6Ef34512b11685dedC509BaDcf"",...",2,2018-11-10 03:32:20,2018,0x00033390560d00f372ae50cD070b8124be5ecE5d,"['0x17559a8138E72C6Ef34512b11685dedC509BaDcf',...",0x00033390560d00f372ae50cD070b8124be5ecE5d,multi_nonowner,2019-10-21 01:04:02+00:00,2019-10-21 03:23:15+00:00,8,344 days 21:31:42,344 days 23:50:55,0 days 02:19:13
1,1,1,0xC3c028721F854BC75967Cbe432fB0e221908Baa1,"[""0x006004fFA18E3cf78fA3b50393ec44C1ab89cF6c"",...",2,2018-12-09 19:41:39,2018,0x0ad59C344359Fdf8472E7FFbf4eB6AF4751138DA,"['0x006004fFA18E3cf78fA3b50393ec44C1ab89cF6c',...",0x0ad59C344359Fdf8472E7FFbf4eB6AF4751138DA,multi_nonowner,2019-02-09 23:17:52+00:00,2019-02-14 20:59:54+00:00,12,62 days 03:36:13,67 days 01:18:15,4 days 21:42:02
2,2,2,0x68326300DF49ec6387E75690857424c2ae111750,"[""0x52DA601C635951a464DA1E38503b7e9C04d7D530"",...",2,2018-12-25 08:34:25,2018,0x5c4c0B176825311452764A2e7327C6b0273b2db2,"['0x52DA601C635951a464DA1E38503b7e9C04d7D530',...",0x5c4c0B176825311452764A2e7327C6b0273b2db2,multi_nonowner,2020-05-05 09:13:47+00:00,2020-05-05 19:14:46+00:00,8,497 days 00:39:22,497 days 10:40:21,0 days 10:00:59
3,3,3,0x5d40522c20326F2Ebcec2D371f250e352E3BED27,"[""0x35EfCe5f4bd52356e8215BCB9dC7687aDe6B6400"",...",7,2019-02-11 16:24:49,2019,0x0aC694A1b86645f2c8563E5fB66a6a22152a694f,"['0x35EfCe5f4bd52356e8215BCB9dC7687aDe6B6400',...",0x0aC694A1b86645f2c8563E5fB66a6a22152a694f,multi_nonowner,2020-01-23 05:10:44+00:00,2020-02-13 23:58:22+00:00,16,345 days 12:45:55,367 days 07:33:33,21 days 18:47:38
4,4,4,0xCa265A7F4C9dc47b259850B696eBeFFA8BB18d9D,"[""0x48143eC3AcA8A255496e8ee7997591D3335Abf5f"",...",2,2019-05-18 19:59:37,2019,0x00806578B4B54D41224cfb1568eFB331DD74e25f,"['0x48143eC3AcA8A255496e8ee7997591D3335Abf5f',...",0x00806578B4B54D41224cfb1568eFB331DD74e25f,multi_nonowner,2020-02-09 18:18:11+00:00,2020-02-11 06:25:22+00:00,4,266 days 22:18:34,268 days 10:25:45,1 days 12:07:11


In [17]:
df_multi_seller.to_csv("../../Data/important/analysis/multiseller_info.csv")

In [59]:
import pandas as pd
import re
from datetime import datetime

def normalize_time_to_standard(s):
    """
    Clean and convert any pool_created_time string (e.g. '2024-11-30T13:18:47Z')
    into the same format as '2018-11-02 22:56:11'
    """
    if pd.isna(s):
        return None

    # Force to string and remove hidden unicode
    s = str(s)
    s = re.sub(r'[^0-9TtZz:+\- ]', '', s).strip()

    # Replace 'T' and 'Z' with proper space/timezone indicator
    s = s.replace('T', ' ').replace('Z', '')

    # Try parsing manually
    for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%d %H:%M:%S%z"):
        try:
            dt = datetime.strptime(s, fmt)
            return dt.strftime("%Y-%m-%d %H:%M:%S")
        except ValueError:
            continue

    # Fallback: let pandas handle any leftovers
    try:
        dt = pd.to_datetime(s, utc=True, errors='coerce')
        if pd.notna(dt):
            return dt.tz_convert(None).strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        pass

    return None  # if all fails

# Apply
df_single_seller['pool_created_time'] = df_single_seller['pool_created_time'].apply(normalize_time_to_standard)

# Verify
print(df_single_seller.loc[[0, 29568], 'pool_created_time'])


0        2018-11-02 22:56:11
29568    2024-11-30 13:18:47
Name: pool_created_time, dtype: object


# Plot and graph and extraction

In [ ]:
# ensure datetime types then compute difference
df_single_seller['first_sell_time'] = pd.to_datetime(df_single_seller['first_sell_time'], utc=True, errors='coerce')
df_single_seller['pool_created_time'] = pd.to_datetime(df_single_seller['pool_created_time'], utc=True, errors='coerce')

df_single_seller['first_sell_delay'] = df_single_seller['first_sell_time'] - df_single_seller['pool_created_time']

# quick check
df_single_seller[['pool_address','pool_created_time','first_sell_time','first_sell_delay']]

In [18]:
df_single_seller

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,pool_address,rugpull_sellers,seller_count,pool_created_time,created_year,token_owner_x,seller_list,seller,tag,token_owner_y,first_sell_time,last_sell_time,sell_count,first_sell_delay,last_sell_delay,sell_span
0,0,0,0,0x4E0E28d426caf318747B8E05C8B0564A580E39a7,"[""0xA8C7372dC993d7510C9c45425807d463967cbb12""]",1,2018-11-02 22:56:11,2018,0xAd850d65eB5202f828f5f7883bc0B46ac87e64D4,['0xA8C7372dC993d7510C9c45425807d463967cbb12'],0xA8C7372dC993d7510C9c45425807d463967cbb12,single-nonowner,0xAd850d65eB5202f828f5f7883bc0B46ac87e64D4,2018-11-03 10:26:39+00:00,2018-11-03 10:26:39+00:00,4,0 days 11:30:28,0 days 11:30:28,0 days 00:00:00
1,1,1,1,0xbaf5A8BDF81cfE2d34c0CeD89236FE473183F2E8,"[""0x8948E4B00DEB0a5ADb909F4DC5789d20D0851D71""]",1,2019-03-07 05:59:20,2019,0x8948E4B00DEB0a5ADb909F4DC5789d20D0851D71,['0x8948E4B00DEB0a5ADb909F4DC5789d20D0851D71'],0x8948E4B00DEB0a5ADb909F4DC5789d20D0851D71,single-owner,0x8948E4B00DEB0a5ADb909F4DC5789d20D0851D71,2019-03-07 18:54:32+00:00,2019-03-07 19:09:30+00:00,4,0 days 12:55:12,0 days 13:10:10,0 days 00:14:58
2,2,2,2,0x225026D626E45FA662e6a71F679efF0CAc3054f1,"[""0x006004fFA18E3cf78fA3b50393ec44C1ab89cF6c""]",1,2019-03-11 14:01:50,2019,0xf1fa9a38914E853DE933FbF7Df2f278701e873DF,['0x006004fFA18E3cf78fA3b50393ec44C1ab89cF6c'],0x006004fFA18E3cf78fA3b50393ec44C1ab89cF6c,single-nonowner,0xf1fa9a38914E853DE933FbF7Df2f278701e873DF,2019-03-12 07:24:20+00:00,2019-03-12 07:24:20+00:00,2,0 days 17:22:30,0 days 17:22:30,0 days 00:00:00
3,3,3,3,0x9394C20adca4512DfC3d3c184c648E4193462Ebb,"[""0x2523C15dB0e3843DfC7C08772c8331Fb40CC8a0F""]",1,2019-04-22 08:02:06,2019,0x2523C15dB0e3843DfC7C08772c8331Fb40CC8a0F,['0x2523C15dB0e3843DfC7C08772c8331Fb40CC8a0F'],0x2523C15dB0e3843DfC7C08772c8331Fb40CC8a0F,single-owner,0x2523C15dB0e3843DfC7C08772c8331Fb40CC8a0F,2019-04-22 18:16:37+00:00,2019-04-25 05:33:33+00:00,8,0 days 10:14:31,2 days 21:31:27,2 days 11:16:56
4,4,4,4,0xEda88dDb13888C9A4dE7304965E9315E69ea980E,"[""0x866cb16F2162c319E48351827A7e15DDf0405E16""]",1,2019-06-10 00:06:22,2019,0x866cb16F2162c319E48351827A7e15DDf0405E16,['0x866cb16F2162c319E48351827A7e15DDf0405E16'],0x866cb16F2162c319E48351827A7e15DDf0405E16,single-owner,0x866cb16F2162c319E48351827A7e15DDf0405E16,2019-06-14 02:09:24+00:00,2019-06-14 20:53:38+00:00,4,4 days 02:03:02,4 days 20:47:16,0 days 18:44:14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29568,29568,29568,29568,0x031D046Cbd8727D15702ECd5299eCcd63ADE84E8,"[""0xC14664811a2a4c233d253fDD03dee4B97ABBEbb5""]",1,2024-11-30 13:18:47,2024,0xCC54E2644ABb3B02EDb21aCD8F4f9ee06432Cada,['0xC14664811a2a4c233d253fDD03dee4B97ABBEbb5'],0xC14664811a2a4c233d253fDD03dee4B97ABBEbb5,single-nonowner,0xCC54E2644ABb3B02EDb21aCD8F4f9ee06432Cada,2024-12-01 17:50:23+00:00,2024-12-01 17:50:23+00:00,2,1 days 04:31:36,1 days 04:31:36,0 days 00:00:00
29569,29569,29569,29569,0xd98d929EBc79856BBaEB783561294999c424211B,"[""0x670C625612D1662c91dc7B4434207101e7bd941f""]",1,2024-11-30 20:59:23,2024,0x670C625612D1662c91dc7B4434207101e7bd941f,['0x670C625612D1662c91dc7B4434207101e7bd941f'],0x670C625612D1662c91dc7B4434207101e7bd941f,single-owner,0x670C625612D1662c91dc7B4434207101e7bd941f,2024-12-01 08:54:11+00:00,2024-12-01 08:54:11+00:00,2,0 days 11:54:48,0 days 11:54:48,0 days 00:00:00
29570,29570,29570,29570,0x6E7FE3428816cbD1CB1788aa84d04C4a08248b02,"[""0x670C625612D1662c91dc7B4434207101e7bd941f""]",1,2024-11-30 21:41:47,2024,0x670C625612D1662c91dc7B4434207101e7bd941f,['0x670C625612D1662c91dc7B4434207101e7bd941f'],0x670C625612D1662c91dc7B4434207101e7bd941f,single-owner,0x670C625612D1662c91dc7B4434207101e7bd941f,2024-12-01 08:56:11+00:00,2024-12-01 08:56:11+00:00,2,0 days 11:14:24,0 days 11:14:24,0 days 00:00:00
29571,29571,29571,29571,0x30600e7a89405e5bD76Be94dABB54d776F93E726,"[""0x3328F7f4A1D1C57c35df56bBf0c9dCAFCA309C49""]",1,2024-11-30 22:27:59,2024,0xEC055397730484b73a9308d24A2A365c86182804,['0x3328F7f4A1D1C57c35df56bBf0c9dCAFCA309C49'],0x3328F7f4A1D1C57c35d

In [ ]:
# ensure datetime types then compute difference
df_multi_seller['first_sell_time'] = pd.to_datetime(df_multi_seller['first_sell_time'], utc=True, errors='coerce')
df_multi_seller['pool_created_time'] = pd.to_datetime(df_multi_seller['pool_created_time'], utc=True, errors='coerce')

df_multi_seller['first_sell_delay'] = df_multi_seller['first_sell_time'] - df_multi_seller['pool_created_time']

# quick check
df_multi_seller[['pool_address','pool_created_time','first_sell_time','first_sell_delay']]

In [81]:

df_single_seller['last_sell_time'] = pd.to_datetime(df_single_seller['last_sell_time'], utc=True, errors='coerce')
df_multi_seller['last_sell_time'] = pd.to_datetime(df_multi_seller['last_sell_time'], utc=True, errors='coerce')
df_single_seller['sell_span'] = df_single_seller['last_sell_time'] - df_single_seller['pool_created_time']
df_multi_seller['sell_span'] = df_single_seller['last_sell_time'] - df_single_seller['pool_created_time']

In [19]:
import pandas as pd
import numpy as np

df = df_single_seller.copy()

# Ensure types
# (If these are already timedeltas, .to_timedelta will be a no-op)
df['first_sell_delay'] = pd.to_timedelta(df['first_sell_delay'], errors='coerce')
df['sell_span']        = pd.to_timedelta(df['sell_span'],        errors='coerce')

# Convert to numeric days for averaging
df['first_sell_delay_days'] = df['first_sell_delay'].dt.total_seconds() / 86400.0
df['sell_span_days']        = df['sell_span'].dt.total_seconds() / 86400.0

# Group by tag and compute means (skip NaN by default)
summary = (
    df.groupby('tag', dropna=False)
      .agg(
          n_pools=('pool_address', 'nunique'),
          avg_sell_count=('sell_count', 'mean'),
          avg_first_sell_delay_days=('first_sell_delay_days', 'mean'),
          avg_sell_span_days=('sell_span_days', 'mean')
      )
      .sort_index()
)

# Optional: nicer rounding
summary = summary.round({
    'avg_sell_count': 2,
    'avg_first_sell_delay_days': 3,
    'avg_sell_span_days': 3
})

print(summary)


                 n_pools  avg_sell_count  avg_first_sell_delay_days  avg_sell_span_days
tag                                                                                    
single-nonowner    17960            3.38                      1.549               0.243
single-owner       11613            3.34                      2.162               0.454


In [ ]:
import pandas as pd
import numpy as np

df = df_single_seller.copy()

# Ensure types
# (If these are already timedeltas, .to_timedelta will be a no-op)
df['first_sell_delay'] = pd.to_timedelta(df['first_sell_delay'], errors='coerce')
df['sell_span']        = pd.to_timedelta(df['sell_span'],        errors='coerce')

# Convert to numeric days for averaging
df['first_sell_delay_days'] = df['first_sell_delay'].dt.total_seconds() / 86400.0
df['sell_span_days']        = df['sell_span'].dt.total_seconds() / 86400.0

# Group by tag and compute means (skip NaN by default)
summary = (
    df.groupby('tag', dropna=False)
      .agg(
          n_pools=('pool_address', 'nunique'),
          avg_sell_count=('sell_count', 'mean'),
          avg_first_sell_delay_days=('first_sell_delay_days', 'mean'),
          avg_sell_span_days=('sell_span_days', 'mean')
      )
      .sort_index()
)

# Optional: nicer rounding
summary = summary.round({
    'avg_sell_count': 2,
    'avg_first_sell_delay_days': 3,
    'avg_sell_span_days': 3
})

print(summary)


In [20]:
import pandas as pd
import numpy as np

df = df_multi_seller.copy()

# Ensure types
# (If these are already timedeltas, .to_timedelta will be a no-op)
df['first_sell_delay'] = pd.to_timedelta(df['first_sell_delay'], errors='coerce')
df['sell_span']        = pd.to_timedelta(df['sell_span'],        errors='coerce')

# Convert to numeric days for averaging
df['first_sell_delay_days'] = df['first_sell_delay'].dt.total_seconds() / 86400.0
df['sell_span_days']        = df['sell_span'].dt.total_seconds() / 86400.0

# Group by tag and compute means (skip NaN by default)
summary = (
    df.groupby('tag', dropna=False)
      .agg(
          n_pools=('pool_address', 'nunique'),
          avg_sell_count=('sell_count', 'mean'),
          avg_first_sell_delay_days=('first_sell_delay_days', 'mean'),
          avg_sell_span_days=('sell_span_days', 'mean')
      )
      .sort_index()
)

# Optional: nicer rounding
summary = summary.round({
    'avg_sell_count': 2,
    'avg_first_sell_delay_days': 3,
    'avg_sell_span_days': 3
})

print(summary)


                n_pools  avg_sell_count  avg_first_sell_delay_days  avg_sell_span_days
tag                                                                                   
multi-owner       23314           82.47                      0.682              11.762
multi_nonowner    52547           96.62                      0.839               6.343


In [21]:
df_scaminfo

,token_created_time,pool_created_time,pool_address,token_paired_address,verified_decimals,token_address,name,unverified_symbol,token_owner,id,fail
0,2018-04-16 00:04:52.000000,2018-11-02 22:56:11,0x4E0E28d426caf318747B8E05C8B0564A580E39a7,0x0000000000000000000000000000000000000000,18.0,0x919D0131fA5F77D99FBBBBaCe50bCb6E62332bf2,BorisCoin,BORIS,0xAd850d65eB5202f828f5f7883bc0B46ac87e64D4,3,False
1,2016-10-24 09:43:52.000000,2018-11-10 03:32:20,0x467fB51D54d7e51eE925F7f1a81AD5f2a0211169,0x0000000000000000000000000000000000000000,18.0,0x888666CA69E0f178DED6D75b5726Cee99A87D698,ICONOMI,ICN,0x00033390560d00f372ae50cD070b8124be5ecE5d,9,False
2,2017-10-19 20:27:35.000000,2018-12-09 19:41:39,0xC3c028721F854BC75967Cbe432fB0e221908Baa1,0x0000000000000000000000000000000000000000,18.0,0x9e88613418cF03dCa54D6a2cf6Ad934A78C7A17A,Swarm Fund Token,SWM,0x0ad59C344359Fdf8472E7FFbf4eB6AF4751138DA,30,False
3,2018-07-12 21:21:00.000000,2018-12-25 08:34:25,0x68326300DF49ec6387E75690857424c2ae111750,0x0000000000000000000000000000000000000000,18.0,0x737fA0372c8D001904Ae6aCAf0552d4015F9c947,MEDIBIT,MEDIBIT,0x5c4c0B176825311452764A2e7327C6b0273b2db2,47,False
4,2018-02-20 09:31:36.000000,2019-02-11 16:24:49,0x5d40522c20326F2Ebcec2D371f250e352E3BED27,0x0000000000000000000000000000000000000000,18.0,0xD49ff13661451313cA1553fd6954BD1d9b6E02b9,ElectrifyAsia,ELEC,0x0aC694A1b86645f2c8563E5fB66a6a22152a694f,79,False
...,...,...,...,...,...,...,...,...,...,...,...
105429,2024-07-08T07:32:59Z,2024-11-30T22:27:59Z,0x30600e7a89405e5bD76Be94dABB54d776F93E726,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,18.0,0x41B6161Fee8dc3AA822b5CBC0ee89e92Ab997a46,SnakeDice,SNAKE,0xEC055397730484b73a9308d24A2A365c86182804,387327,False
105430,2024-11-30T14:20:59Z,2024-11-30T23:01:11Z,0x6555fD6e6a1f048030314a26c5BFdfb42c9D3512,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,18.0,0x855F652bcA1AEa2D62A422620333162A3bbEbCa0,TOP HAT,HAT,0xf3c739AC6db9258D6152CBA121a96Fb9F5B7Efd8,387331,False
105431,2024-11-30T21:27:11Z,2024-11-30T23:01:47Z,0xa89f5Fefa693418d6A40ef68366493BaAA5c8212,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,18.0,0x3f7dB133aFf2F012C8534b36aB9731fe9Ee7bd43,Monerochan,MONEROCHAN,0x22748a8FfCC62940cbAb2f800E39832E16088C6E,387333,False
105432,2024-11-30T23:05:59Z,2024-11-30T23:05:59Z,0x9474F4C5b7a491cf223c960Cf52D60B147acBE17,0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2,18.0,0x749e4b7b6d92C57c9E910D61b4B434d770cAF2B5,PEPE GRINCH,PGRINCH,0x0E971EFD97F74054c5E1EBf2cA613ecCa5C7bF23,387335,False


In [4]:


# combine single and multi into a single table `mix_seller`
cols = ['pool_address', 'seller_count', 'sell_count']

s_single = df_single_seller[cols].copy()
s_multi = df_multi_seller[cols].copy()

# concat with multi last so multi rows override single when same pool_address exists
mix = pd.concat([s_single, s_multi], ignore_index=True)

# keep one row per pool_address, prefer the last occurrence (multi overrides single)
mix = mix.drop_duplicates(subset='pool_address', keep='last').reset_index(drop=True)

# normalize types
mix['seller_count'] = pd.to_numeric(mix['seller_count'], errors='coerce').fillna(0).astype(int)
mix['sell_count']   = pd.to_numeric(mix['sell_count'], errors='coerce').astype('Int64')

# final table
mix_seller = mix[['pool_address', 'seller_count', 'sell_count']]

# show sample
mix_seller.head()

,pool_address,seller_count,sell_count
0,0x4E0E28d426caf318747B8E05C8B0564A580E39a7,1,4
1,0xbaf5A8BDF81cfE2d34c0CeD89236FE473183F2E8,1,4
2,0x225026D626E45FA662e6a71F679efF0CAc3054f1,1,2
3,0x9394C20adca4512DfC3d3c184c648E4193462Ebb,1,8
4,0xEda88dDb13888C9A4dE7304965E9315E69ea980E,1,4


In [28]:
mix_seller

,pool_address,seller_count,sell_count
0,0x4E0E28d426caf318747B8E05C8B0564A580E39a7,1,4
1,0xbaf5A8BDF81cfE2d34c0CeD89236FE473183F2E8,1,4
2,0x225026D626E45FA662e6a71F679efF0CAc3054f1,1,2
3,0x9394C20adca4512DfC3d3c184c648E4193462Ebb,1,8
4,0xEda88dDb13888C9A4dE7304965E9315E69ea980E,1,4
...,...,...,...
105429,0x80777279644b44cBA3F5EC23B4AACDf76b106793,45,272
105430,0x6fFd814BBDd0c7D0Ab4050030f2536EC7AE6B0Ba,30,286
105431,0xa89f5Fefa693418d6A40ef68366493BaAA5c8212,50,764
105432,0x9474F4C5b7a491cf223c960Cf52D60B147acBE17,21,116


In [5]:
import numpy as np
import pandas as pd

df = mix_seller.copy()
x = pd.to_numeric(df['seller_count'], errors='coerce')
y = pd.to_numeric(df['sell_count'],   errors='coerce')

# ---- Coarse, interpretable bins ----
x_edges  = [1, 2, 5, 10, np.inf]                 # -> {1}, {2–4}, {5–9}, {10+}
x_labels = ["1", "2–4", "5–9", "10+"]

y_edges  = [0, 10, 50, 250, np.inf]              # -> {0–9}, {10–49}, {50–249}, {250+}
y_labels = ["0–9", "10–49", "50–249", "250+"]

gx = pd.cut(x, bins=x_edges, labels=x_labels, right=False, include_lowest=True)
gy = pd.cut(y, bins=y_edges, labels=y_labels, right=False, include_lowest=True)

# 2D table and shares
tab = pd.crosstab(gx, gy, dropna=False).rename_axis(index='Wallets', columns='Sells')
total = tab.values.sum()
share = (tab / total * 100).round(2)

# Show the coarse table with counts and %
print("Counts:\n", tab, "\n")
print("Share (%):\n", share, "\n")

# Top 6 coarse cells by share
flat = (tab.stack().rename('count').to_frame())
flat['share_pct'] = (flat['count'] / total * 100).round(2)
print("Top coarse cells:\n", flat.sort_values('share_pct', ascending=False).head(6))

# If you want to define 3–4 big categories, combine adjacent cells:
# Example categories (tweak to taste):
cat_defs = {
    "Compact drains":             (gx.isin(["1","2–4"])  & gy.isin(["0–9","10–49"])),
    "Staged mid-volume":          (gx.isin(["2–4","5–9"]) & gy.isin(["50–249"])),
    "Distributed campaigns":      (gx.isin(["10+"])      & gy.isin(["50–249","250+"])),
    "High-churn wide teams":      (gx.isin(["5–9","10+"]) & gy.isin(["250+"])),
}

cat_rows = []
for name, mask in cat_defs.items():
    cnt = int(mask.sum())
    pct = round(cnt / total * 100, 2)
    cat_rows.append((name, cnt, pct))

cat_summary = pd.DataFrame(cat_rows, columns=["Category", "Count", "Share %"]).sort_values("Share %", ascending=False)
print("\nCategory summary (coarse, 3–4 bins):\n", cat_summary)


Counts:
 Sells      0–9  10–49  50–249  250+
Wallets                            
NaN        116      0       0     0
1        28395   1088      86     4
2–4      17017  11245     767    39
5–9          4  11843    1881   122
10+          0   4073   22740  6014 

Share (%):
 Sells      0–9  10–49  50–249  250+
Wallets                            
NaN       0.11   0.00    0.00  0.00
1        26.93   1.03    0.08  0.00
2–4      16.14  10.67    0.73  0.04
5–9       0.00  11.23    1.78  0.12
10+       0.00   3.86   21.57  5.70 

Top coarse cells:
                 count  share_pct
Wallets Sells                   
1       0–9     28395      26.93
10+     50–249  22740      21.57
2–4     0–9     17017      16.14
5–9     10–49   11843      11.23
2–4     10–49   11245      10.67
10+     250+     6014       5.70

Category summary (coarse, 3–4 bins):
                 Category  Count  Share %
0         Compact drains  57745    54.77
2  Distributed campaigns  28754    27.27
3  High-churn wide teams  

In [21]:
import os

# ------------------------------------------------------------------
# 0) Data prep (change this if your merged frame has a different name)
# ------------------------------------------------------------------
df = df = mix_seller.copy()

# Ensure numeric
df['seller_count'] = pd.to_numeric(df['seller_count'], errors='coerce')
df['sell_count']   = pd.to_numeric(df['sell_count'],   errors='coerce')

# Drop rows with missing values in the two core columns
df = df.dropna(subset=['seller_count', 'sell_count'])

# ------------------------------------------------------------------
# 1) Define three main categories (as agreed)
#    - Minimal Drains: 1 wallet, 0–9 sells
#    - Distributed Campaigns: 10+ wallets, 50–249 sells
#    - Moderate Networks: 2–9 wallets, <= 50 sells
#    - Other: everything else
# ------------------------------------------------------------------
minimal_mask    = (df['seller_count'] == 1) & (df['sell_count'].between(0, 9))
distributed_mask= (df['seller_count'] >= 10) & (df['sell_count'].between(50, 249))
moderate_mask   = (df['seller_count'].between(2, 9)) & (df['sell_count'] <= 50)

df['category'] = 'Other LPs'
df.loc[minimal_mask,     'category'] = 'Minimal Drains LPs'
df.loc[distributed_mask, 'category'] = 'Distributed Campaigns LPs'
df.loc[moderate_mask,    'category'] = 'Moderate Networks LPs'

# Quick summary (shares)
cat_summary = (
    df['category'].value_counts(dropna=False)
      .rename_axis('category')
      .to_frame('count')
)
cat_summary['share_pct'] = (cat_summary['count'] / len(df) * 100).round(2)
print(cat_summary)

# ------------------------------------------------------------------
# 2) Axis caps (99th percentile) so outliers don’t dominate the view
# ------------------------------------------------------------------
x_cap = int(np.ceil(df['seller_count'].quantile(0.99)))
y_cap = int(np.ceil(df['sell_count'].quantile(0.99)))
print({'x_cap_99pct': x_cap, 'y_cap_99pct': y_cap})

# Clipped views for plotting
x_clip = df['seller_count'].clip(lower=1, upper=x_cap)
y_clip = df['sell_count'].clip(lower=0, upper=y_cap)

# ------------------------------------------------------------------
# 3) Scatter plot by category (others light gray)
# ------------------------------------------------------------------
os.makedirs('figures', exist_ok=True)

fig, ax = plt.subplots(figsize=(7, 5))

# Plot Others first (light gray) for background, small points
mask_other = df['category'].eq('Other LPs')
ax.scatter(
    x_clip[mask_other], y_clip[mask_other],
    s=5, alpha=0.15, color='#A9A9A9', label='Other LPs'
)

# Then overlay the three main groups with distinct colors
palette = {
    'Minimal Drains LPs':          '#1f77b4',  # blue
    'Distributed Campaigns LPs':   '#ff7f0e',  # orange
    'Moderate Networks LPs':       '#9467bd',  # purple
}

for cat, color in palette.items():
    m = df['category'].eq(cat)
    if m.any():
        ax.scatter(
            x_clip[m], y_clip[m],
            s=9, alpha=0.7, color=color, label=f'{cat} (n={m.sum():,})'
        )

ax.set_xlabel('Number of seller wallets', fontsize=18)
ax.set_ylabel('Total number of sells', fontsize=18)
ax.set_xlim(0.5, x_cap + 0.5)
ax.set_ylim(-1, y_cap + y_cap*0.03)
ax.tick_params(axis='both', labelsize=15)
ax.grid(True, alpha=0.3)
ax.legend(frameon=True, loc='upper right', fontsize=14)
plt.tight_layout()
plt.savefig('figures/wallet_sell_scatter.pdf', bbox_inches='tight')
plt.close(fig)
plt.show()
print('Saved: figures/wallet_sell_scatter.pdf')

# ------------------------------------------------------------------
# 4) Density hexbin (supports the scatter)
# ------------------------------------------------------------------
# fig, ax = plt.subplots(figsize=(7, 5))
# hb = ax.hexbin(
#     x_clip, y_clip,
#     gridsize=45, mincnt=1, linewidths=0.0
# )
# ax.set_xlabel('Number of seller wallets')
# ax.set_ylabel('Total number of sells')
# ax.set_xlim(0.5, x_cap + 0.5)
# ax.set_ylim(-1, y_cap + y_cap*0.03)
# cb = fig.colorbar(hb, ax=ax)
# cb.set_label('Pool count')
# ax.grid(True, alpha=0.2)
# # plt.tight_layout()
# # plt.savefig('figures/wallet_sell_density.pdf', bbox_inches='tight')
# # plt.close(fig)
# plt.show()
# print('Saved: figures/wallet_sell_density.pdf')


                           count  share_pct
category                                   
Moderate Networks LPs      40271      38.20
Minimal Drains LPs         28395      26.93
Distributed Campaigns LPs  22740      21.57
Other LPs                  14028      13.31
{'x_cap_99pct': 117, 'y_cap_99pct': 742}
Saved: figures/wallet_sell_scatter.pdf
